# RAVDESS (video) -> audio .wav -> spectrogrammes RGB + CSV de metadonnees

Telecharge les zip video de RAVDESS (speech + song, un par acteur), extrait l'audio des
fichiers full-AV (modalite 01) en .wav via ffmpeg, supprime les fichiers video-only
(modalite 02, sans son), puis convertit en segments de 3s en mel-spectrogramme RGB.
S'ajoute a la suite dans le meme `data/labels.csv` et `data/spectrograms/labels.csv`.

In [ ]:
%pip install -q librosa matplotlib numpy pandas soundfile pillow nbstripout imageio-ffmpeg

import sys
!{sys.executable} -m nbstripout --install --attributes .gitattributes

In [ ]:
import io, zipfile, urllib.request
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
import soundfile as sf
from matplotlib import cm
from PIL import Image

VIDEO_URLS = [
    f"https://zenodo.org/records/1188976/files/Video_Speech_Actor_{i:02d}.zip?download=1"
    for i in range(1, 25)
] + [
    f"https://zenodo.org/records/1188976/files/Video_Song_Actor_{i:02d}.zip?download=1"
    for i in range(1, 25) if i != 18
]
URLS = VIDEO_URLS
DATA_DIR = Path("data")
RAW = DATA_DIR / "ravdess-video"

OUT = Path("data/spectrograms")
LABELS_CSV = Path("data/labels.csv")

DATASET_NAME = "RAVDESS"

# colonnes communes a toutes les bases (union des taxonomies)
EMOTIONS_ALL = [
    "anger", "boredom", "calm", "disgust", "fear",
    "happiness", "neutral", "sadness", "surprise",
]
RAVDESS_EMOTIONS = {"anger", "calm", "disgust", "fear", "happiness", "neutral", "sadness", "surprise"}

# 3eme partie du nom de fichier (emotion)
EMOTION_CODES = {
    "01": "neutral", "02": "calm", "03": "happiness", "04": "sadness",
    "05": "anger", "06": "fear", "07": "disgust", "08": "surprise",
}

In [ ]:
# 1. telechargement + extraction (~24 Go, ca prend du temps)
if not RAW.exists():
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    for url in URLS:
        print("downloading...", url)
        data = urllib.request.urlopen(url).read()
        zipfile.ZipFile(io.BytesIO(data)).extractall(RAW)

    inner_zips = list(RAW.rglob("*.zip"))
    while inner_zips:
        for inner in inner_zips:
            zipfile.ZipFile(inner).extractall(inner.parent)
            inner.unlink()
        inner_zips = list(RAW.rglob("*.zip"))

In [ ]:
# 1b. video -> audio .wav (modalite 01 = full-AV, a un son) ; modalite 02 = video-only, sans son -> supprime
import subprocess
import imageio_ffmpeg

FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()

for mp4 in RAW.rglob("*.mp4"):
    modality = mp4.stem.split("-")[0]
    wav = mp4.with_suffix(".wav")
    if modality == "01":
        try:
            subprocess.run(
                [FFMPEG, "-y", "-i", str(mp4), "-vn", "-ac", "1", "-ar", "16000", str(wav)],
                check=True, capture_output=True,
            )
        except subprocess.CalledProcessError:
            pass
    mp4.unlink()

wavs = sorted(RAW.rglob("*.wav"))
print(len(wavs), "fichiers wav")

In [ ]:
# 2. metadonnees (parsees depuis le nom de fichier) -> 1 CSV (accumule sur plusieurs bases)
rows = []
for wav in wavs:
    modality, vocal, emo_code, intensity, statement, rep, actor = wav.stem.split("-")
    actor = int(actor)
    rows.append({
        "file": wav.name,
        "subdir": str(wav.relative_to(RAW).parent),
        "speaker": actor,
        "age": None,
        "gender": "male" if actor % 2 == 1 else "female",
        "language": "eng",
        "emotion.confidence": None,
        "emotion.naturalness": None,
        "transcription": statement,
        "duration": round(sf.info(wav).duration, 3),
        "emotion": EMOTION_CODES[emo_code],
    })
df = pd.DataFrame(rows)

if LABELS_CSV.exists():
    existing = pd.read_csv(LABELS_CSV, na_values=["None"])
    dup = existing.merge(df.assign(root=str(RAW))[["root", "file"]], on=["root", "file"], how="inner")
    if len(dup):
        raise ValueError(f"{len(dup)} fichiers de {DATASET_NAME} deja presents dans {LABELS_CSV}")
    same = existing[existing["dataset"] == DATASET_NAME]
    prior_speaker = dict(zip(same["initial_speaker"], same["speaker"]))
    prior_transcript = dict(zip(same["initial_transcript"], same["transcript"]))
    next_speaker = int(existing["speaker"].max()) + 1
    next_transcript = int(existing["transcript"].max()) + 1
else:
    existing = None
    prior_speaker = {}
    prior_transcript = {}
    next_speaker = 1
    next_transcript = 1

df["initial_speaker"] = df["speaker"]
speaker_map = {}
for code in sorted(df["speaker"].unique()):
    if code in prior_speaker:
        speaker_map[code] = prior_speaker[code]
    else:
        speaker_map[code] = next_speaker
        next_speaker += 1
df["speaker"] = df["speaker"].map(speaker_map)

df["initial_transcript"] = df["transcription"]
transcript_map = {}
for code in sorted(df["transcription"].unique()):
    if code in prior_transcript:
        transcript_map[code] = prior_transcript[code]
    else:
        transcript_map[code] = next_transcript
        next_transcript += 1
df["transcript"] = df["transcription"].map(transcript_map)
df = df.drop(columns=["transcription"])

for emo in EMOTIONS_ALL:
    df[emo] = (df["emotion"] == emo).astype(int) if emo in RAVDESS_EMOTIONS else None
df = df.drop(columns=["emotion"])

df.insert(0, "dataset", DATASET_NAME)
df.insert(0, "root", str(RAW))

cols = ["root", "dataset", "subdir", "file", "duration", "speaker", "initial_speaker",
        "age", "gender", "language", "emotion.confidence", "emotion.naturalness",
        "transcript", "initial_transcript"] + EMOTIONS_ALL
df = df[cols]

if existing is not None:
    df = pd.concat([existing, df], ignore_index=True)

df.to_csv(LABELS_CSV, index=False, na_rep="None")
print(len(df), "lignes dans", LABELS_CSV)

In [ ]:
# 3. wav -> segments de 3s -> mel-spectrogramme -> image RGB (accumule sur plusieurs bases)
SR = 16000
DURATION = 3.0
MIN_SEG = 1.0
N_FFT = int(0.025 * SR)
HOP = int(0.010 * SR)
N_MELS = 64
N_FRAMES = int(DURATION * SR / HOP)
CHUNK = int(DURATION * SR)
MIN_SAMPLES = int(MIN_SEG * SR)

SPEC_CSV = OUT / "labels.csv"
label_by_file = dict(zip(df["file"], df[EMOTIONS_ALL].fillna(0).idxmax(axis=1)))
meta = df.set_index("file").to_dict(orient="index")

spec_existing = pd.read_csv(SPEC_CSV, na_values=["None"]) if SPEC_CSV.exists() else None
done = set(zip(spec_existing["root"], spec_existing["base_file"])) if spec_existing is not None else set()

def mel_to_rgb(y):
    y = librosa.util.fix_length(y, size=CHUNK)
    mel = librosa.feature.melspectrogram(
        y=y, sr=SR, n_fft=N_FFT, hop_length=HOP, win_length=N_FFT, n_mels=N_MELS
    )
    db = librosa.power_to_db(mel)[:, :N_FRAMES]
    z = (db - db.mean()) / (db.std() + 1e-9)
    norm = np.clip(z, -3, 3) / 6 + 0.5
    rgb = (cm.viridis(norm[::-1])[..., :3] * 255).astype(np.uint8)
    return Image.fromarray(rgb)

rows = []
for wav in wavs:
    if (str(RAW), wav.name) in done:
        continue
    y, _ = librosa.load(wav, sr=SR, mono=True)
    label = label_by_file[wav.name]

    for n, start in enumerate(range(0, len(y), CHUNK)):
        seg = y[start:start + CHUNK]
        if len(seg) < MIN_SAMPLES:
            continue
        name = f"{wav.stem}_{n}"
        dest = OUT / label
        dest.mkdir(parents=True, exist_ok=True)
        mel_to_rgb(seg).save(dest / f"{name}.png")
        rows.append({
            "spectrogram": f"{label}/{name}.png",
            "base_file": wav.name,
            "chunk": n,
            **meta[wav.name],
            "duration": round(sf.info(wav).duration, 3),
        })

new_spec = pd.DataFrame(rows)
spec_df = pd.concat([spec_existing, new_spec], ignore_index=True) if spec_existing is not None else new_spec
spec_df.to_csv(SPEC_CSV, index=False, na_rep="None")
print(len(rows), "nouveaux spectrogrammes,", len(spec_df), "total dans", SPEC_CSV)

In [ ]:
# 4. verification rapide
for d in sorted(OUT.iterdir()):
    if d.is_dir():
        print(d.name, len(list(d.glob("*.png"))))

display(Image.open(next(OUT.rglob("*.png"))))

import csv as _csv
with LABELS_CSV.open(encoding="utf-8") as f:
    for i, row in zip(range(5), _csv.reader(f)):
        print(row)